<a href="https://colab.research.google.com/github/nsandoval-uagro/CalidadAguaVT/blob/main/ML_Calidad_del_Agua.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# -----------------------------
# 1) Importaciones
# -----------------------------
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

from sklearn.model_selection import KFold, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import joblib

# XGBoost Opcional
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False


# -----------------------------
# 2) Configuración (Parámetros del proyecto)
# -----------------------------
@dataclass
class CONFIG:
    """Clase para centralizar todas las rutas de archivos y parámetros del modelo."""
    # Entrada
    in_csv: str = "/content/drive/MyDrive/Datos CA VT/BD Completa.csv"

    # Salidas
    out_dir: str = "/content/drive/MyDrive/OUT_WQ_PIPELINE"
    out_dataset_csv: str = "BD_Completa_mejorada_features.csv"
    out_results_csv: str = "resultados_modelos_multiparametro.csv"

    # Variables objetivo
    targets: Tuple[str, ...] = (
        "pH", "Temperatura", "Conductividad", "Eh",
        "Nitratos", "Sulfatos", "Floruros", "Alcalinidad", "Cloruros"
    )

    # Coordenadas (opcional para validación cruzada espacial)
    x_col: str = "X"
    y_col: str = "Y"
    use_spatial_groups: bool = True
    grid_size: float = 120.0  # metros si es UTM

    # Bandas espectrales base esperadas en el CSV
    base_bands: Tuple[str, ...] = (
        "Banda_2", "Banda_3", "Banda_4", "Banda_5",
        "Banda_6", "Banda_7", "Banda_8", "Banda_8a",
        "Banda_11", "Banda_12"
    )

    # Ratios controlados (evita la explosión de variables)
    ratios: Tuple[Tuple[str, str], ...] = (
        ("Banda_3", "Banda_5"),   # proxy verde / red-edge
        ("Banda_2", "Banda_5"),   # proxy azul / red-edge
        ("Banda_4", "Banda_2"),   # ratio rojo / azul
        ("Banda_8", "Banda_4"),   # ratio nir / rojo
        ("Banda_11", "Banda_12"), # swir1 / swir2
        ("Banda_7", "Banda_8a"),  # red-edge / red-edge
        ("Banda_5", "Banda_7"),   # ratio red-edge
        ("Banda_2", "Banda_4"),   # azul / rojo
    )

    # Diferencias normalizadas
    normdiffs: Tuple[Tuple[str, str], ...] = (
        ("Banda_3", "Banda_5"),
        ("Banda_8", "Banda_4"),
    )

    # Preprocesamiento
    drop_zero_targets: bool = True
    min_samples_target: int = 20

    # Validación Cruzada (CV)
    outer_splits: int = 5
    inner_splits: int = 4
    random_state: int = 42

    # Exportación
    export_best_per_target: bool = True


cfg = CONFIG()
os.makedirs(cfg.out_dir, exist_ok=True)


# -----------------------------
# 3) Utilidades (Funciones de soporte)
# -----------------------------
def safe_div(a: pd.Series, b: pd.Series, eps: float = 1e-9) -> pd.Series:
    """Realiza una división segura evitando la división por cero."""
    return a / (b.replace(0, np.nan) + eps)

def safe_normdiff(a: pd.Series, b: pd.Series, eps: float = 1e-9) -> pd.Series:
    """Calcula la diferencia normalizada (a-b)/(a+b)."""
    return (a - b) / (a + b + eps)

def make_spatial_groups(df: pd.DataFrame, x_col: str, y_col: str, grid_size: float) -> pd.Series:
    """Crea grupos espaciales basados en una cuadrícula para evitar autocorrelación en la validación."""
    gx = (df[x_col] / grid_size).round().astype("Int64")
    gy = (df[y_col] / grid_size).round().astype("Int64")
    return (gx.astype(str) + "_" + gy.astype(str)).astype("string")

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """Calcula métricas estándar de regresión (R2, RMSE, MAE, Sesgo)."""
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": rmse,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "Bias": float(np.mean(y_pred - y_true)),
    }

def summarize_cv(scores: List[Dict[str, float]]) -> Dict[str, float]:
    """Agrega los resultados de múltiples pliegues de validación cruzada."""
    out = {}
    for k in scores[0].keys():
        arr = np.array([s[k] for s in scores], dtype=float)
        out[f"{k}_mean"] = float(np.mean(arr))
        out[f"{k}_std"] = float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0
    return out


# -----------------------------
# 4) E/S + Limpieza de datos
# -----------------------------
def load_and_clean(csv_path: str) -> pd.DataFrame:
    """Carga el CSV y realiza una limpieza inicial de nombres y valores infinitos."""
    df = pd.read_csv(csv_path)
    df.columns = df.columns.astype(str).str.strip()
    df = df.replace([np.inf, -np.inf], np.nan)
    return df


# -----------------------------
# 5) Ingeniería de variables (Controlada)
# -----------------------------
def build_features(df: pd.DataFrame, cfg: CONFIG) -> pd.DataFrame:
    """Genera ratios e índices espectrales basados en la configuración."""
    feats = df.copy()
    bands = [b for b in cfg.base_bands if b in feats.columns]

    for b1, b2 in cfg.ratios:
        if b1 in feats.columns and b2 in feats.columns:
            name = f"ratio_{b1}_div_{b2}"
            feats[name] = safe_div(feats[b1], feats[b2])

    for b1, b2 in cfg.normdiffs:
        if b1 in feats.columns and b2 in feats.columns:
            name = f"nd_{b1}_minus_{b2}_over_sum"
            feats[name] = safe_normdiff(feats[b1], feats[b2])

    created = [c for c in feats.columns if c.startswith("ratio_") or c.startswith("nd_")]
    feature_cols = bands + created
    feats[feature_cols] = feats[feature_cols].apply(pd.to_numeric, errors="coerce")
    feats.attrs["feature_cols"] = feature_cols
    return feats


# -----------------------------
# 6) Modelos + Grillas de hiperparámetros
# -----------------------------
def get_models_and_grids(cfg: CONFIG) -> Dict[str, Tuple[object, Dict]]:
    """Define los modelos y los rangos de búsqueda para optimización."""
    models = {}
    models["Ridge"] = (Ridge(random_state=cfg.random_state), {"regressor__alpha": [0.1, 1.0, 10.0, 50.0, 100.0]})
    models["ElasticNet"] = (ElasticNet(random_state=cfg.random_state, max_iter=25000), {"regressor__alpha": [0.001, 0.01, 0.1, 1.0, 10.0], "regressor__l1_ratio": [0.1, 0.5, 0.9]})
    models["SVR_RBF"] = (SVR(kernel="rbf"), {"regressor__C": [1, 10, 100, 300], "regressor__epsilon": [0.01, 0.1, 0.3], "regressor__gamma": ["scale", 0.1, 1.0]})
    models["RF"] = (RandomForestRegressor(random_state=cfg.random_state), {"regressor__n_estimators": [300, 600], "regressor__max_depth": [None, 8, 16], "regressor__min_samples_leaf": [1, 2, 4], "regressor__max_features": ["sqrt", 0.5]})
    if HAS_XGB:
        models["XGB"] = (XGBRegressor(random_state=cfg.random_state, n_estimators=600, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0, max_depth=6, objective="reg:squarederror", n_jobs=-1), {"regressor__max_depth": [3, 5, 7], "regressor__min_child_weight": [1, 5], "regressor__subsample": [0.7, 0.9], "regressor__colsample_bytree": [0.7, 0.9]})
    return models


# -----------------------------
# 7) Evaluación por Validación Cruzada Anidada
# -----------------------------
def nested_cv_evaluate(X: pd.DataFrame, y: pd.Series, groups: Optional[pd.Series], model_name: str, base_estimator, param_grid: Dict, cfg: CONFIG) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    """Ejecuta una validación cruzada anidada para asegurar una evaluación sin sesgos."""
    base_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("regressor", base_estimator)])
    model = TransformedTargetRegressor(regressor=base_pipe, func=None, inverse_func=None)

    if groups is not None:
        outer = GroupKFold(n_splits=cfg.outer_splits)
        outer_splits = outer.split(X, y, groups=groups)
    else:
        outer = KFold(n_splits=cfg.outer_splits, shuffle=True, random_state=cfg.random_state)
        outer_splits = outer.split(X, y)

    y_oof, yhat_oof = np.full(len(y), np.nan), np.full(len(y), np.nan)
    fold_scores = []

    for tr_idx, te_idx in outer_splits:
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

        if groups is not None:
            g_tr = groups.iloc[tr_idx]
            inner_cv = GroupKFold(n_splits=min(cfg.inner_splits, g_tr.nunique())).split(X_tr, y_tr, groups=g_tr)
        else:
            inner_cv = KFold(n_splits=cfg.inner_splits, shuffle=True, random_state=cfg.random_state)

        gs = GridSearchCV(estimator=model, param_grid={f"regressor__{k}": v for k, v in param_grid.items()}, scoring="r2", cv=inner_cv, n_jobs=-1)
        gs.fit(X_tr, y_tr)
        y_pred = gs.best_estimator_.predict(X_te)
        y_oof[te_idx], yhat_oof[te_idx] = y_te.values, y_pred
        fold_scores.append(regression_metrics(y_te.values, y_pred))

    return summarize_cv(fold_scores), y_oof, yhat_oof


# -----------------------------
# 8) Gráficos de diagnóstico (Opcionales)
# -----------------------------
def plot_real_vs_pred(y_true: np.ndarray, y_pred: np.ndarray, title: str):
    """Genera un gráfico de dispersión de valores Reales vs Predichos."""
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.6)
    mn, mx = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    plt.plot([mn, mx], [mn, mx], "k--")
    plt.xlabel("Real"); plt.ylabel("Predicho"); plt.title(title); plt.grid(True); plt.show()

def plot_residuals(y_true: np.ndarray, y_pred: np.ndarray, title: str):
    """Genera gráficos para el análisis de los residuos (errores)."""
    resid = y_true - y_pred
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1); plt.scatter(y_pred, resid, alpha=0.6); plt.axhline(0, ls="--"); plt.title(f"{title} - Residuos")
    plt.subplot(1, 2, 2); plt.hist(resid, bins=30, edgecolor="k"); plt.title(f"{title} - Histograma"); plt.show()


# -----------------------------
# 9) Función Principal (Main)
# -----------------------------
def main(cfg: CONFIG):
    print("=== Cargando datos ===")
    df = load_and_clean(cfg.in_csv)
    feats = build_features(df, cfg)
    feature_cols = feats.attrs["feature_cols"]

    groups = make_spatial_groups(feats, cfg.x_col, cfg.y_col, cfg.grid_size) if cfg.use_spatial_groups else None
    X = feats[feature_cols].copy().fillna(feats[feature_cols].median(numeric_only=True))

    all_results = []
    models = get_models_and_grids(cfg)

    for target in cfg.targets:
        if target not in feats.columns: continue
        y = pd.to_numeric(feats[target], errors="coerce")
        valid = y.notna() & (y != 0 if cfg.drop_zero_targets else True)
        if valid.sum() < cfg.min_samples_target: continue

        print(f"\nTarget: {target} | n={valid.sum()}")
        for name, (est, grid) in models.items():
            summary, y_oof, yhat_oof = nested_cv_evaluate(X.loc[valid], y.loc[valid], groups.loc[valid] if groups is not None else None, name, est, grid, cfg)
            summary["target"] = target
            all_results.append(summary)
            print(f"- {name:8s} | R2={summary['R2_mean']:.3f}")

    results_df = pd.DataFrame(all_results).sort_values(["target", "R2_mean"], ascending=[True, False])
    results_df.to_csv(os.path.join(cfg.out_dir, cfg.out_results_csv), index=False)
    print("\nProceso finalizado.")


# -----------------------------
# 10) Ejecución del script
# -----------------------------
if __name__ == "__main__":
    main(cfg)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Cargando datos ===

Target: pH | n=36
- Ridge    | R2=0.056
- ElasticNet | R2=-0.171
- SVR_RBF  | R2=-0.169
- RF       | R2=0.062
- XGB      | R2=-0.996

Target: Temperatura | n=36
- Ridge    | R2=-0.000
- ElasticNet | R2=-0.007
- SVR_RBF  | R2=0.267
- RF       | R2=0.130
- XGB      | R2=-0.269

Target: Conductividad | n=36
- Ridge    | R2=0.053
- ElasticNet | R2=-0.085
- SVR_RBF  | R2=-0.490
- RF       | R2=0.171
- XGB      | R2=-0.388

Target: Eh | n=36
- Ridge    | R2=-1.256
- ElasticNet | R2=-0.905
- SVR_RBF  | R2=-0.377
- RF       | R2=-1.274
- XGB      | R2=-1.620

Target: Nitratos | n=25
- Ridge    | R2=0.090
- ElasticNet | R2=-0.097
- SVR_RBF  | R2=0.027
- RF       | R2=0.018
- XGB      | R2=-0.427

Target: Sulfatos | n=36
- Ridge    | R2=-0.942
- ElasticNet | R2=-0.807
- SVR_RBF  | R2=-0.735
- RF       | R2=-1.177
- XGB      | R2=-1.367

Target: 